# Notebook 13 — Training Data, Tokenization, and Packing

    ## Learning objectives

    - Build reproducible dataset splits without leakage
- Format conversational examples with the model chat template
- Compare padding, concatenation, packing, truncation, and loss masking

    Cells labeled **optional GPU/remote** are deliberately guarded. Read them first,
    then opt in when the required hardware or Hugging Face Inference access is available.


In [ ]:
# Colab/local environment setup — run this cell first.
import importlib.util
import os
import platform
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules
PACKAGES = ['transformers>=4.51,<5', 'datasets>=3.5,<6', 'sentencepiece']

if IN_COLAB and PACKAGES:
    print("Installing notebook dependencies in the Colab runtime...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *PACKAGES])

# Load an HF token from Colab Secrets without displaying it. In Colab, create a
# secret named HF_TOKEN (or HUGGINGFACE_TOKEN) and enable notebook access.
if IN_COLAB:
    from google.colab import userdata
    token = None
    for secret_name in ("HF_TOKEN", "HUGGINGFACE_TOKEN"):
        try:
            token = userdata.get(secret_name)
        except Exception:
            pass
        if token:
            break
    if token:
        os.environ["HF_TOKEN"] = token
        os.environ["HUGGINGFACE_TOKEN"] = token
else:
    try:
        from dotenv import load_dotenv
        load_dotenv(".env")
    except ImportError:
        pass

try:
    import torch
    accelerator = torch.cuda.get_device_name(0) if torch.cuda.is_available() else (
        "Apple MPS" if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available() else "CPU"
    )
    print(f"runtime={platform.platform()} | Python={platform.python_version()} | accelerator={accelerator}")
    if False and not torch.cuda.is_available():
        print("WARNING: this training notebook is designed for a Colab GPU runtime. "
              "Select Runtime > Change runtime type > T4 GPU (or better).")
except ImportError:
    print(f"runtime={platform.platform()} | Python={platform.python_version()}")

print("Hugging Face token configured:", bool(os.getenv("HUGGINGFACE_TOKEN")))


## 13.1 Data quality defines the objective

Pretraining predicts all eligible tokens. SFT usually trains on conversational text,
sometimes masking user/system tokens so loss applies only to assistant responses.
Deduplicate before splitting; near-duplicates across train and evaluation inflate scores.
Track provenance, license, language, safety filtering, and dataset version.


In [ ]:
from datasets import Dataset
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-0.5B-Instruct")
rows = [
    {"messages": [{"role": "user", "content": "What is 2+2?"},
                  {"role": "assistant", "content": "4"}]},
    {"messages": [{"role": "user", "content": "Define entropy briefly."},
                  {"role": "assistant", "content": "A measure of uncertainty in a distribution."}]},
]
ds = Dataset.from_list(rows)
rendered = [tokenizer.apply_chat_template(r["messages"], tokenize=False) for r in rows]
for text in rendered: print(repr(text))


In [ ]:
max_length = 64
tokenized = tokenizer(rendered, truncation=True, max_length=max_length,
                      padding="max_length", return_tensors="pt")
labels = tokenized["input_ids"].clone()
labels[tokenized["attention_mask"] == 0] = -100
utilization = tokenized["attention_mask"].float().mean().item()
print("shape:", tokenized["input_ids"].shape)
print(f"non-padding utilization: {utilization:.1%}")


## 13.2 Packing and contamination

Packing combines short examples into full sequences to reduce padding. Boundaries need
EOS tokens, correct position handling, and deliberate attention behavior: examples may
attend across boundaries unless block-diagonal masking is used. Truncation can silently
delete answers or image tokens. Inspect length distributions before choosing limits.


## 13.3 Dataset lifecycle and governance

Treat a training dataset as a versioned software artifact. Record immutable source IDs,
acquisition dates, licenses/terms, consent and privacy constraints, filtering code,
deduplication method, language/domain labels, and cryptographic fingerprints. Keep raw,
cleaned, formatted, tokenized, and split stages distinct so a bug can be traced rather than
silently baked into a final Arrow file.

Exact hash deduplication catches copies after normalization; MinHash or embedding methods
detect near-duplicates. Deduplicate before splitting, preferably at document/source family
level. Random row splits leak templated variants and adjacent chunks. For future-facing
applications, time-based evaluation is more honest. For users or organizations, group splits
test generalization and prevent identity leakage. Search training data for benchmark prompts
and reference answers, not merely dataset names.

Filtering is modeling: removing profanity, code, minority dialects, short responses, or
refusals changes behavior. Document intended and unintended distribution changes. Manually
inspect stratified samples before and after every major filter.


In [ ]:
# Small deterministic dataset audit utilities.
import hashlib, re
def normalize(text):
    return re.sub(r"\s+", " ", text.strip().lower())
def fingerprint(text):
    return hashlib.sha256(normalize(text).encode()).hexdigest()[:12]

audit_rows = [
    {"source": "a", "text": "Gradient accumulation uses microbatches."},
    {"source": "b", "text": " gradient   accumulation uses microbatches. "},
    {"source": "c", "text": "Activation checkpointing recomputes activations."},
]
seen = {}
for row in audit_rows:
    key = fingerprint(row["text"])
    print(row["source"], key, "duplicate_of", seen.get(key))
    seen.setdefault(key, row["source"])


## 13.4 Chat templates and assistant-only loss

A conversational dataset is structured records, not preformatted strings. Preserve roles
and content, then render with the target tokenizer's chat template. Templates determine BOS,
role delimiters, EOS placement, generation prompts, and sometimes tool syntax. Training with
one template and serving with another is distribution shift. Confirm a round trip on multi-
turn, system, tool, empty, and long examples.

Assistant-only loss requires knowing which rendered tokens belong to assistant messages.
Searching decoded text for a delimiter is fragile because delimiters can appear in content
and token boundaries differ. Prefer templates that return assistant masks or a data collator
designed for the exact template. Decide whether assistant headers and EOS are targets. Tool
calls and reasoning fields may require distinct policy. Print tokens alongside masks for a
handful of examples before training.


In [ ]:
# Inspect every rendered token; adapt the target column to the template's mask support.
sample = rows[0]["messages"]
rendered_ids = tokenizer.apply_chat_template(sample, tokenize=True,
                                              add_generation_prompt=False)
print("idx | id | token")
for i, token_id in enumerate(rendered_ids):
    piece = tokenizer.decode([token_id]).replace("\n", "\\n")
    print(f"{i:3d} | {token_id:6d} | {piece!r}")


## 13.5 Length policy, packing, and throughput

Plot token lengths after final rendering. Choose maximum length using percentile coverage,
task requirements, available memory, and truncation semantics. “Keep end” may preserve the
answer but lose the question; “keep start” can delete the answer. Filter or construct windows
deliberately when neither is safe. For long documents, sample spans or pack naturally rather
than always taking prefixes.

Packing raises utilization but changes boundaries. Standard causal packing allows later
examples to attend to earlier ones, even if loss is separated by EOS. Block-diagonal
attention prevents contamination but needs compatible kernels/collators. Position IDs may
reset or continue. Sequence packing also changes batch-length variance and tokens/update.
Report effective *tokens* per optimizer step, not only examples. Dynamic padding plus length
bucketing is a simpler intermediate optimization.

**Preflight reference:** schema validation; role alternation; nonempty assistant targets;
special-token correctness; length/truncation report; duplicate/leakage report; masked-token
percentage; packed utilization; and decoded random samples from the actual dataloader.


## 13.6 Training-data reference

| Stage | Required evidence |
|---|---|
| Source | Provenance, license/consent, timestamp, immutable ID |
| Cleaning | Transformation/filter versions and before/after samples |
| Deduplication | Normalization, exact/near-duplicate method, cluster IDs |
| Split | Group/time/source policy and leakage audit |
| Formatting | Target chat template/tokenizer revision |
| Tokenization | Length distribution, truncation, special tokens |
| Labels | Assistant/role masks and target-token percentage |
| Packing | Boundary/EOS, attention and position policy, utilization |

Data bugs often look like optimizer bugs. Inspect the actual collated batch: decoded IDs, attention
mask, labels with ignored tokens, position IDs, and shifted targets. Calculate tokens/update and
source mixture after sampling—not only raw dataset proportions. Keep a small “golden batch” whose
rendered tokens and masks are regression-tested whenever tokenizer/template/collator versions change.

Privacy and deletion requirements propagate into derived chunks, tokenized caches, checkpoints, and
logs. Dataset documentation should state known gaps and filtering harms, not only row counts.


## 13.6 Packing without cross-document leakage

Concatenating documents improves token utilization but changes the learning problem unless boundaries are explicit. Insert EOS tokens, decide whether attention may cross boundaries, and decide whether the first token after a boundary should be predicted from the prior document. Sequence packing can use block-diagonal attention and per-segment position IDs, but kernel support varies. Compare padding waste, packed utilization, and boundary transitions on real length distributions. Preserve document and license metadata through tokenization so deletion, filtering, and audits remain possible after packing.


In [ ]:
lengths=torch.tensor([3,7,2,6]); max_length=8
padded_tokens=len(lengths)*max_length; useful=int(lengths.sum()); print("padding utilization",useful/padded_tokens)
bins=[]
for length in sorted(lengths.tolist(),reverse=True):
 for b in bins:
  if sum(b)+length<=max_length: b.append(length); break
 else: bins.append([length])
print("packed bins",bins,"utilization",useful/(len(bins)*max_length))


## 13.7 Data-mixture sampling

Sampling datasets in proportion to raw size can erase small high-value domains; uniform sampling can overrepeat tiny sources. Define mixture weights, temperature sampling, caps, and curriculum phases in token terms. Log realized rather than requested proportions after filtering, tokenization, and worker sharding. Evaluate slices aligned with each source and watch for memorization under repetition. Data weighting is an optimization decision and a governance decision: retain source, license, consent, quality, language, and deduplication lineage for every example.


In [ ]:
sizes=torch.tensor([1_000_000.,100_000.,10_000.])
for temperature in (1.0,.5,0.0):
 weights=torch.ones_like(sizes) if temperature==0 else sizes.pow(temperature); weights/=weights.sum(); print(temperature,weights.tolist())


## Exercises

    1. Measure padding waste with and without length bucketing.
2. Implement a greedy best-fit packing function and preserve EOS boundaries.
3. Write five automated dataset checks, including leakage and empty responses.

    ## Checkpoint

    Explain the notebook's central mechanism without using library names, then identify
    one assumption you would test before applying it to a real workload.
